# CivicConnect AI — Multilingual Complaint Classification

Supervised ML using word + character TF-IDF and Logistic Regression for English, Hindi and Marathi civic complaints.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
df = pd.read_csv('complaint_dataset.csv')
print(df.shape)
df.head()

## Dataset
The curated academic dataset contains 540 labeled complaints: 180 English, 180 Hindi and 180 Marathi examples across six categories.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['category'], test_size=0.20, random_state=42, stratify=df['category']
)


In [ ]:
model = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1,2), sublinear_tf=True)),
        ('char_tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2,5), sublinear_tf=True))
    ])),
    ('classifier', LogisticRegression(max_iter=2500, class_weight='balanced', random_state=42))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


In [ ]:
tests = [
    'There is a large pothole near Central Market.',
    'मुख्य बाजार के पास सड़क पर बड़ा गड्ढा है।',
    'मुख्य बाजाराजवळ रस्त्यावर मोठा खड्डा पडला आहे.'
]
for text in tests:
    probs = model.predict_proba([text])[0]
    i = probs.argmax()
    print(text, '=>', model.classes_[i], round(probs[i]*100, 2), '%')


## Deployment
The trained pipeline is saved as `complaint_classifier.joblib` and loaded by FastAPI. Every new complaint is classified by the model at runtime.

Limitation: the dataset is curated for academic demonstration; production deployment would require a much larger real-world labeled dataset.